# Cinemática Inversa Numérica

In [ ]:
import numpy as np
from numpy import float32, float64, int32, int64
from dataclasses import dataclass
from typing import Self

: 

## Versión inicial inexacta

In [ ]:
@dataclass
class Pose:
  rotacion: np.ndarray
  translacion: np.ndarray

  @classmethod
  def identidad(cls) -> Pose:
    return Pose(
      np.eye(3), 
      np.zeros(3),
    )

  def __mul__(self, pose: Self) -> Pose:
    return Pose(
      self.rotacion @ pose.rotacion,
      self.rotacion @ pose.translacion + self.translacion,
    )

  def distancia_cuadrado(self, pose: Self) -> float64:
    diff = self.translacion - pose.translacion
    return diff[0]**2 + diff[1]**2 + diff[2]**2

def DH(theta: float64, d: float64, a: float64, alpha: float64) -> Pose:
    cth = np.cos(theta)
    sth = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    return Pose(np.array([
        [cth, -sth*ca,  sth*sa],
        [sth,  cth*ca, -cth*sa],
        [  0,     sa,       ca],
    ]), np.array([ a*cth, a*sth, d ]))

In [ ]:
@dataclass
class Dimensiones:
  d1: float64
  a1: float64
  a2: float64
  d4: float64
  d6: float64

class IRB140:
  def __init__(self, dim: Dimensiones, ABase0: Pose | None = None, A6Tool: Pose | None = None):
    self.A01 = lambda q1: DH(q1, dim.d1, dim.a1, -np.pi/2)
    self.A12 = lambda q2: DH(q2,      0, dim.a2,        0)
    self.A23 = lambda q3: DH(q3,      0,      0,  np.pi/2)
    self.A34 = lambda q4: DH(q4, dim.d4,      0, -np.pi/2)
    self.A45 = lambda q5: DH(q5,      0,      0,  np.pi/2)
    self.A56 = lambda q6: DH(q6, dim.d6,      0,        0)

    self.ABase0 = Pose.identidad() if ABase0 is None else ABase0
    self.A6Tool = Pose.identidad() if A6Tool is None else A6Tool

  def cinematica_directa(self, q: np.ndarray) -> Pose:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())
    A06 = self.A01(q1) * self.A12(q2) * self.A23(q3) * self.A34(q4) * self.A45(q5) * self.A56(q6)
    return self.ABase0 * A06 * self.A6Tool


In [ ]:
def funcion_costo(pose_origen: Pose, pose_destino: Pose) -> np.float64:
  return pose_origen.distancia_cuadrado(pose_destino)

In [ ]:
irb140 = IRB140(Dimensiones(352., 70, 360, 380, 65))

In [ ]:
def cinematima_inversa(pose_destino: Pose, alfa: float64, epsilon: float64 = 1e-3, max_iteraciones: int32 = 1000) -> np.ndarray:
  q = np.array([0., 0., 0., 0., 0., 0.])

  for _ in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      return q

    gradiente = np.zeros(6)
    for i in range(6):
      variacion = 0.01

      delta_q = np.zeros(6)
      delta_q[i] += variacion

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / variacion

    q -= alfa * gradiente

  print("Llegó a max iteraciones")
  return q

pose_destino = irb140.cinematica_directa(np.array([0, -np.pi/2, np.pi, 0, np.pi/2, 0]))
cinematima_inversa(pose_destino, 0.001, max_iteraciones = 10000)